<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

In [1]:
# TODO 0: 실습을 위해 아래 패키지를 import 해주세요.
import numpy as np
import seaborn as sns
import ssl

ssl._create_default_https_context = ssl._create_unverified_context
sns.set_theme()

</br>

# 학습 내용
>이번 장에서는 <strong>데이터 정규화(Data Normalization)</strong>에 대해 학습합니다.</br></br>
>수치형 데이터 선별과 표준화·정규화 기법을 NumPy로 직접 학습해봅시다.

</br>

# 수치형 데이터 선별 (Numeric Feature Selection)
> 정규화 전에 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">수치형 열만 선택</mark>해야 합니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">include 값</th>
      <th style="text-align:center">선택되는 열</th>
      <th style="text-align:center">예시 dtype</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>"number"</code></td><td style="text-align:center">모든 수치형 열</td><td style="text-align:center"><code>int64</code>, <code>float64</code></td></tr>
    <tr><td style="text-align:center"><code>"int"</code></td><td style="text-align:center">정수형 열만</td><td style="text-align:center"><code>int32</code>, <code>int64</code></td></tr>
    <tr><td style="text-align:center"><code>"float"</code></td><td style="text-align:center">실수형 열만</td><td style="text-align:center"><code>float32</code>, <code>float64</code></td></tr>
    <tr><td style="text-align:center"><code>"object"</code></td><td style="text-align:center">문자열(범주형) 열</td><td style="text-align:center"><code>object</code></td></tr>
    <tr><td style="text-align:center"><code>"category"</code></td><td style="text-align:center">카테고리형 열</td><td style="text-align:center"><code>CategoricalDtype</code></td></tr>
    <tr><td style="text-align:center"><code>"bool"</code></td><td style="text-align:center">불리언 열</td><td style="text-align:center"><code>bool</code></td></tr>
    <tr><td style="text-align:center"><code>"datetime"</code></td><td style="text-align:center">날짜/시간 열</td><td style="text-align:center"><code>datetime64</code></td></tr>
  </tbody>
</table>

💡 `include`에 리스트로 여러 타입을 전달할 수 있습니다.
> 예: `df.select_dtypes(include=["int", "float"])`

In [ ]:
# TODO 1:
#  - seaborn의 diamonds 데이터 셋을 df에 저장해봅시다.
#  - 수치형 열(number)만 선택하여 numeric_df 변수에 저장한 뒤, 전체 열과 수치형 열을 출력해봅시다.
# seaborn은 주로 sns로 약어를 만들어서 사용함

df = sns.load_dataset("diamonds")

numeric_df = df.select_dtypes(include=["number"]) # number, 수치만 들어잇는 걸로 변환
print(df.columns)
print(numeric_df.columns)

Index(['carat', 'cut', 'color', 'clarity', 'depth', 'table', 'price', 'x', 'y',
       'z'],
      dtype='object')
Index(['carat', 'depth', 'table', 'price', 'x', 'y', 'z'], dtype='object')


# carat은 실수 price는 정수 
# 숫자 차이가 크면 ex 0.23 - 326만 이렇게 차이가 크면 학습할때 불리하다. -> 표준화가 필요하다.

</br>

# 표준화 (Standardization)
> 데이터를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">평균 0, 표준편차 1</mark>로 변환하는 스케일링 기법입니다.

</br>

## 왜 스케일링이 필요한가?

> 피처마다 **단위와 범위가 다르면** 모델이 특정 피처에 편향됩니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">피처</th>
      <th style="text-align:center">범위</th>
      <th style="text-align:center">단위</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>carat</code></td><td style="text-align:center">0.2 ~ 5.01</td><td style="text-align:center">캐럿</td></tr>
    <tr><td style="text-align:center"><code>price</code></td><td style="text-align:center">326 ~ 18,823</td><td style="text-align:center">달러</td></tr>
  </tbody>
</table>

> 경사하강법에서 스케일이 큰 피처가 손실 함수의 기울기를 지배하여, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습이 느려지거나 발산</mark>할 수 있습니다.</br>
> 스케일링을 통해 모든 피처를 **동일한 기준**으로 맞추면 학습이 안정적이고 빠르게 수렴합니다.

</br>

## 평균(Mean)과 표준편차(Standard Deviation)

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">통계량</th>
      <th>수식</th>
      <th>의미</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align:center">평균 $\mu$</td>
      <td style="text-align:center">$\displaystyle\mu = \frac{1}{m}\sum_{i=1}^{m} x_i$</td>
      <td style="text-align:center">데이터의 <strong>중심 위치</strong></td>
    </tr>
    <tr>
      <td style="text-align:center">표준편차 $\sigma$</td>
      <td style="text-align:center">$\displaystyle\sigma = \sqrt{\frac{1}{m}\sum_{i=1}^{m}(x_i - \mu)^2}$</td>
      <td style="text-align:center">데이터가 평균으로부터 <strong>얼마나 퍼져 있는지</strong></td>
    </tr>
  </tbody>
</table>

> 표준화는 각 값에서 평균을 빼서 **중심을 0으로** 옮기고, 표준편차로 나누어 **퍼짐 정도를 1로** 통일합니다.

</br>

$$z = \frac{x - \mu}{\sigma}$$

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">기호</th>
      <th style="text-align:center">의미</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">$x$</td><td style="text-align:center">원본 값</td></tr>
    <tr><td style="text-align:center">$\mu$</td><td style="text-align:center">평균 (mean)</td></tr>
    <tr><td style="text-align:center">$\sigma$</td><td style="text-align:center">표준편차 (standard deviation)</td></tr>
    <tr><td style="text-align:center">$z$</td><td style="text-align:center">표준화된 값 (z-score)</td></tr>
  </tbody>
</table>

💡axis 매개변수
> `axis=0`은 각 **열**에 대해, `axis=1`은 각 **행**에 대해 연산합니다.</br>
> 정규화에서는 각 피처(열)별로 통계값을 구해야 하므로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">axis=0</mark>을 사용합니다.

In [11]:
# TODO 2: numeric_df를 NumPy 배열로 변환하고, 평균과 표준편차를 구한 뒤, 표준화 공식으로 변환을 수행해봅시다.

data = numeric_df.values # 원본 배열

mean = np.mean(data, axis=0) # 평균 구하기 mean
std = np.std(data, axis=0) # 표준편차 std

z = (data-mean) / std # 표준화 공식 

print(data)

print(mean)

print(std)

print(z)

[[ 0.23 61.5  55.   ...  3.95  3.98  2.43]
 [ 0.21 59.8  61.   ...  3.89  3.84  2.31]
 [ 0.23 56.9  65.   ...  4.05  4.07  2.31]
 ...
 [ 0.7  62.8  60.   ...  5.66  5.68  3.56]
 [ 0.86 61.   58.   ...  6.15  6.12  3.74]
 [ 0.75 62.2  55.   ...  5.83  5.87  3.64]]
[7.97939748e-01 6.17494049e+01 5.74571839e+01 3.93279972e+03
 5.73115721e+00 5.73452595e+00 3.53873378e+00]
[4.74006851e-01 1.43260804e+00 2.23446985e+00 3.98940276e+03
 1.12175035e+00 1.14212409e+00 7.05692305e-01]
[[-1.19816781 -0.17409151 -1.09967199 ... -1.58783745 -1.53619556
  -1.57112919]
 [-1.24036129 -1.36073849  1.58552871 ... -1.64132529 -1.65877419
  -1.74117497]
 [-1.19816781 -3.38501862  3.37566251 ... -1.49869105 -1.45739502
  -1.74117497]
 ...
 [-0.20662095  0.73334442  1.13799526 ... -0.06343409 -0.04774083
   0.03013526]
 [ 0.13092691 -0.52310533  0.24292836 ...  0.37338325  0.33750627
   0.28520393]
 [-0.10113725  0.31452784 -1.09967199 ...  0.08811478  0.11861587
   0.14349912]]


💡표준화를 사용하는 경우
> 이상치가 있어도 비교적 안정적이며, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정규분포에 가까운 데이터</mark>에 적합합니다.</br>
> 경사하강법 기반 모델(선형 회귀, 신경망)에서 학습 속도를 크게 개선합니다.

</br>

# 정규화 (Min-Max Normalization)
> 데이터를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">[0, 1] 범위</mark>로 변환하는 스케일링 기법입니다.

💡왜 [0, 1] 범위로 변환하는가?
> KNN은 데이터 간 **거리**로 이웃을 판단합니다. 스케일이 다르면 거리가 왜곡됩니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">피처</th>
      <th style="text-align:center">A</th>
      <th style="text-align:center">B</th>
      <th style="text-align:center">차이</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>carat</code></td><td style="text-align:center">0.23</td><td style="text-align:center">0.31</td><td style="text-align:center">0.08</td></tr>
    <tr><td style="text-align:center"><code>price</code></td><td style="text-align:center">326</td><td style="text-align:center">18,823</td><td style="text-align:center">18,497</td></tr>
  </tbody>
</table>

> `carat` 차이는 0.08이지만 `price` 차이는 18,497로, 거리 계산에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">price가 거의 100% 지배</mark>합니다.</br>
> [0, 1]로 정규화하면 두 피처가 동등한 비중으로 거리에 반영됩니다.

$$x' = \frac{x - x_{min}}{x_{max} - x_{min}}$$

In [ ]:
# TODO 3: 공식을 직접 구현하여 Min-Max 정규화를 수행해봅시다.

x_min = np.min(data, axis=0) # 최솟값
x_max = np.max(data, axis=0) # 최댓값

x = (data-x_min) / (x_max-x_min) # 0과 1 사이의 값으로 바뀜

print(x_min)

print(x_max)

print(x) 

[2.00e-01 4.30e+01 4.30e+01 3.26e+02 0.00e+00 0.00e+00 0.00e+00]
[5.0100e+00 7.9000e+01 9.5000e+01 1.8823e+04 1.0740e+01 5.8900e+01
 3.1800e+01]
[[0.00623701 0.51388889 0.23076923 ... 0.36778399 0.06757216 0.07641509]
 [0.002079   0.46666667 0.34615385 ... 0.36219739 0.06519525 0.07264151]
 [0.00623701 0.38611111 0.42307692 ... 0.37709497 0.06910017 0.07264151]
 ...
 [0.1039501  0.55       0.32692308 ... 0.52700186 0.09643463 0.11194969]
 [0.13721414 0.5        0.28846154 ... 0.5726257  0.10390492 0.11761006]
 [0.11434511 0.53333333 0.23076923 ... 0.54283054 0.09966044 0.11446541]]


💡스케일링 주의
> 학습 데이터의 통계값(평균, 최소, 최대)으로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">테스트 데이터도 변환</mark>해야 합니다.</br>
> 테스트 데이터의 통계값을 별도로 계산하면 데이터 누수가 발생합니다.